# 보관용 공개 사본

당시 팀 프로젝트의 구현 자료입니다. 출력과 개인 공유 경로를 정리했습니다. 데이터·학습 가중치는 포함하지 않으며, 현재 환경에서 전체 실행을 검증하지 않았습니다. 상세한 버전 차이와 실행 조건은 `docs/evidence.md`, `docs/reproduction.md`를 참고하세요.

원본에는 `trainer` 생성 전 평가 호출과 중복 실험 셀이 남아 있습니다. 이 사본은 당시 코드를 보존하며 위에서 아래로 전체 실행이 가능한 정리본이라고 주장하지 않습니다.


In [ ]:
# -*- coding: utf-8 -*-
"""
Model B (동조/sycophancy 멀티태스크) 학습 스크립트 - 수정본

[수정 사항]
1. (치명적) 데이터 경로: train_processed_v3.csv / valid_processed_v3.csv (Model A용, sycophancy=0 placeholder)
   -> sycophancy_train.csv / sycophancy_val.csv (Model B 전용, 질문 단위로 분할된 실제 동조 데이터)로 교체
2. 모델 태그: beomi/KcELECTRA-base-v2022 -> beomi/KcELECTRA-base (팀 합의 규격으로 통일)
3. evaluation_strategy -> eval_strategy (transformers 5.x 최신 규격)
4. class weight 계산 함수를 make_full_weights 방식으로 교체
   - 특정 클래스가 배치/분할 안에서 0개가 되어도 인덱스 길이가 항상 num_classes로 고정되도록 안전장치 추가
"""

!pip install -q transformers datasets accelerate scikit-learn pandas

import os
import random
import numpy as np
import torch
from transformers import set_seed
from google.colab import drive

# 1. Seed 고정
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. 구글 드라이브 마운트
drive.mount('/content/drive')

# 3. 실제 폴더 경로 지정 (공유 드라이브 바로가기로 확인된 실제 경로)
# 주의: "AI 윤리 모니터링 모델" 뒤에 공백이 하나 더 있음 (실제 폴더명 그대로)
PROJECT_DIR = "/content/drive/MyDrive/voda-sentinel/syco_model"
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print("현재 작업 경로 :", os.getcwd())
    print("폴더 파일 목록:", os.listdir())
else:
    raise FileNotFoundError(f"❌ 경로를 찾을 수 없습니다: {PROJECT_DIR}")

2

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight

MODEL_NAME = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_path = None
val_path = None
possible_train_paths = [
    'data/processed/sycophancy_train.csv',
    'data/02_processed/sycophancy_train.csv',
    'data/sycophancy_train.csv',
    'sycophancy_train.csv',
]
for p in possible_train_paths:
    if os.path.exists(p):
        train_path = p
        break

possible_val_paths = [
    'data/processed/sycophancy_val.csv',
    'data/02_processed/sycophancy_val.csv',
    'data/sycophancy_val.csv',
    'sycophancy_val.csv',
]
for p in possible_val_paths:
    if os.path.exists(p):
        val_path = p
        break

if train_path and val_path:
    print(f"✨ Train 파일 발견 및 로드: {train_path}")
    print(f"✨ Val 파일 발견 및 로드: {val_path}")
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
else:
    raise FileNotFoundError("❌ 'sycophancy_train.csv' 및 'sycophancy_val.csv' 파일을 찾을 수 없습니다.")

syc_unique = sorted(train_df['sycophancy'].unique().tolist())
if syc_unique != [0, 1]:
    raise ValueError(f"❌ sycophancy 라벨이 이상합니다: {syc_unique}.")
print(f"✅ sycophancy 라벨 확인: {syc_unique} (정상)")

train_df['labels'] = train_df.apply(lambda x: [x['sycophancy'], x['bias_level'], x['severity']], axis=1)
val_df['labels'] = val_df.apply(lambda x: [x['sycophancy'], x['bias_level'], x['severity']], axis=1)

train_dataset = Dataset.from_pandas(train_df[['text', 'labels']], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[['text', 'labels']], preserve_index=False)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def make_full_weights(labels, num_classes, boost=None):
    present = np.unique(labels)
    w = compute_class_weight('balanced', classes=present, y=labels)
    full = np.ones(num_classes, dtype=np.float32)
    for c, val in zip(present, w):
        full[int(c)] = val
    if boost:
        for c, mult in boost.items():
            full[c] *= mult
    return torch.tensor(full, dtype=torch.float)

class_weights = {
    "syc": make_full_weights(train_df['sycophancy'].values, num_classes=2),
    "bias": make_full_weights(train_df['bias_level'].values, num_classes=4),
    "sev": make_full_weights(train_df['severity'].values, num_classes=3),
}
print("✅ Tokenization 및 Class Weights 계산 완료!")
print("  syc weights:", class_weights["syc"].numpy().round(3))
print("  bias weights:", class_weights["bias"].numpy().round(3))
print("  sev weights:", class_weights["sev"].numpy().round(3))

3

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    syc_logits, bias_logits, sev_logits = predictions

    pred_syc = np.argmax(syc_logits, axis=-1)
    pred_bias = np.argmax(bias_logits, axis=-1)
    pred_sev = np.argmax(sev_logits, axis=-1)

    true_syc = labels[:, 0]
    true_bias = labels[:, 1]
    true_sev = labels[:, 2]

    metrics = {}
    for name, true, pred in [
        ("syc", true_syc, pred_syc),
        ("bias", true_bias, pred_bias),
        ("sev", true_sev, pred_sev),
    ]:
        metrics[f"{name}_acc"] = accuracy_score(true, pred)
        metrics[f"{name}_prec"] = precision_score(true, pred, average='macro', zero_division=0)
        metrics[f"{name}_rec"] = recall_score(true, pred, average='macro', zero_division=0)
        metrics[f"{name}_f1"] = f1_score(true, pred, average='macro', zero_division=0)

    metrics["overall_macro_f1"] = (metrics["syc_f1"] + metrics["bias_f1"] + metrics["sev_f1"]) / 3

    return metrics

4

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids
    syc_logits, bias_logits, sev_logits = predictions
    pred_syc = np.argmax(syc_logits, axis=-1)
    pred_bias = np.argmax(bias_logits, axis=-1)
    pred_sev = np.argmax(sev_logits, axis=-1)
    true_syc = labels[:, 0]
    true_bias = labels[:, 1]
    true_sev = labels[:, 2]
    metrics = {}
    for name, true, pred in [("syc", true_syc, pred_syc), ("bias", true_bias, pred_bias), ("sev", true_sev, pred_sev)]:
        metrics[f"{name}_acc"] = accuracy_score(true, pred)
        metrics[f"{name}_prec"] = precision_score(true, pred, average='macro', zero_division=0)
        metrics[f"{name}_rec"] = recall_score(true, pred, average='macro', zero_division=0)
        metrics[f"{name}_f1"] = f1_score(true, pred, average='macro', zero_division=0)
    metrics["overall_macro_f1"] = (metrics["syc_f1"] + metrics["bias_f1"] + metrics["sev_f1"]) / 3
    return metrics

In [ ]:
eval_result = trainer.evaluate()
print(eval_result)

5

In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Optional, Tuple
from transformers.modeling_outputs import ModelOutput
from transformers import ElectraModel, ElectraPreTrainedModel, Trainer, TrainingArguments


@dataclass
class MultiTaskOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: Optional[Tuple[torch.FloatTensor, torch.FloatTensor, torch.FloatTensor]] = None
    syc_logits: Optional[torch.FloatTensor] = None
    bias_logits: Optional[torch.FloatTensor] = None
    sev_logits: Optional[torch.FloatTensor] = None


class KcELECTRAMultiTaskModel(ElectraPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.electra = ElectraModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.syc_classifier = nn.Linear(config.hidden_size, 2)
        self.bias_classifier = nn.Linear(config.hidden_size, 4)
        self.sev_classifier = nn.Linear(config.hidden_size, 3)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        outputs = self.electra(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = self.dropout(outputs.last_hidden_state[:, 0, :])
        syc_logits = self.syc_classifier(pooled_output)
        bias_logits = self.bias_classifier(pooled_output)
        sev_logits = self.sev_classifier(pooled_output)
        return MultiTaskOutput(
            syc_logits=syc_logits,
            bias_logits=bias_logits,
            sev_logits=sev_logits,
        )   # logits=(...) 필드는 제거 — predictions 언패킹 시 4개로 잡히는 문제 방지


class MultiTaskTrainer(Trainer):
    def __init__(self, class_weights=None, loss_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.loss_weights = loss_weights or {"syc": 1.0, "bias": 0.5, "sev": 0.5}

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"].long()      # pop() 아니고 인덱싱 (평가 루프에서 labels 다시 읽을 수 있도록)
        labels_syc = labels[:, 0]
        labels_bias = labels[:, 1]
        labels_sev = labels[:, 2]

        outputs = model(**inputs)             # labels는 forward의 **kwargs로 들어가서 자동 무시됨

        syc_logits = outputs.syc_logits
        bias_logits = outputs.bias_logits
        sev_logits = outputs.sev_logits
        curr_device = syc_logits.device

        loss_fct_syc = nn.CrossEntropyLoss(weight=self.class_weights["syc"].to(curr_device))
        loss_fct_bias = nn.CrossEntropyLoss(weight=self.class_weights["bias"].to(curr_device))
        loss_fct_sev = nn.CrossEntropyLoss(weight=self.class_weights["sev"].to(curr_device))

        loss_syc = loss_fct_syc(syc_logits.view(-1, 2), labels_syc.view(-1))
        loss_bias = loss_fct_bias(bias_logits.view(-1, 4), labels_bias.view(-1))
        loss_sev = loss_fct_sev(sev_logits.view(-1, 3), labels_sev.view(-1))

        loss = (
            self.loss_weights["syc"] * loss_syc +
            self.loss_weights["bias"] * loss_bias +
            self.loss_weights["sev"] * loss_sev
        )
        return (loss, outputs) if return_outputs else loss

In [ ]:
eval_result = trainer.evaluate()
print(eval_result)

6

In [ ]:
from transformers import AutoConfig, EarlyStoppingCallback

MODEL_NAME = "beomi/KcELECTRA-base"

config = AutoConfig.from_pretrained(MODEL_NAME)
model = KcELECTRAMultiTaskModel.from_pretrained(MODEL_NAME, config=config)

LOSS_WEIGHTS = {
    "syc": 1.0,
    "bias": 0.5,
    "sev": 0.5,
}

training_args = TrainingArguments(
    output_dir="./models/v1_checkpoint",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_overall_macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    class_weights=class_weights,
    loss_weights=LOSS_WEIGHTS,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# 학습 전 사전 점검: compute_metrics가 실제로 채워지는지 먼저 확인
print("🔎 사전 점검 (초기 가중치 상태 evaluate)...")
sanity_check = trainer.evaluate()
print(sanity_check)

print("\n🚀 Model B(동조/sycophancy) 멀티태스크 학습을 시작합니다...")
trainer.train()

FINAL_MODEL_PATH = "./models/sycophancy_multitask_best"
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"✅ 최종 모델 저장 완료: {FINAL_MODEL_PATH}")

테스트

In [ ]:
import pandas as pd

model.eval()
device = next(model.parameters()).device

results = []
for i in range(len(val_df)):
    text = str(val_df.iloc[i]['text'])
    true_syc = int(val_df.iloc[i]['sycophancy'])

    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = model(**enc)
    pred_syc = out.syc_logits.argmax(-1).item()
    prob_syc = torch.softmax(out.syc_logits, dim=-1)[0, 1].item()  # sycophancy=1일 확률

    results.append({
        "text": text,
        "true_syc": true_syc,
        "pred_syc": pred_syc,
        "prob_syc1": round(prob_syc, 3),
        "text_len": len(text),
    })

result_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 100)
print(result_df.sort_values("prob_syc1", ascending=False).to_string())

In [ ]:
print(val_df.groupby('sycophancy')['text'].apply(lambda x: x.str.len().describe()))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X_train_len = train_df['text'].str.len().values.reshape(-1, 1)
y_train_syc = train_df['sycophancy'].values
X_val_len = val_df['text'].str.len().values.reshape(-1, 1)
y_val_syc = val_df['sycophancy'].values

clf = LogisticRegression().fit(X_train_len, y_train_syc)
pred = clf.predict(X_val_len)
print("문장 길이만으로 syc_f1:", f1_score(y_val_syc, pred, average='macro'))

In [ ]:
def starts_with_agreement(text):
    # [AI 답변] 뒤 첫 부분만 추출
    ai_part = text.split("[AI 답변]")[-1].strip()
    agree_markers = ["네,", "네.", "맞습니다"]
    return 1 if any(ai_part.startswith(m) for m in agree_markers) else 0

val_df['keyword_pred'] = val_df['text'].apply(starts_with_agreement)
from sklearn.metrics import f1_score
print("키워드(첫 어절)만으로 syc_f1:", f1_score(val_df['sycophancy'], val_df['keyword_pred'], average='macro'))

In [ ]:
import os
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Optional, Tuple
import pandas as pd
from transformers import AutoConfig, AutoTokenizer, ElectraModel, ElectraPreTrainedModel
from transformers.modeling_outputs import ModelOutput
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from google.colab import files

# ==========================================
# 1. 모델 클래스 정의 (NameError 방지)
# ==========================================
@dataclass
class MultiTaskOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: Optional[Tuple[torch.FloatTensor, torch.FloatTensor, torch.FloatTensor]] = None
    syc_logits: Optional[torch.FloatTensor] = None
    bias_logits: Optional[torch.FloatTensor] = None
    sev_logits: Optional[torch.FloatTensor] = None


class KcELECTRAMultiTaskModel(ElectraPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.electra = ElectraModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.syc_classifier = nn.Linear(config.hidden_size, 2)
        self.bias_classifier = nn.Linear(config.hidden_size, 4)
        self.sev_classifier = nn.Linear(config.hidden_size, 3)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        outputs = self.electra(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = self.dropout(outputs.last_hidden_state[:, 0, :])
        syc_logits = self.syc_classifier(pooled_output)
        bias_logits = self.bias_classifier(pooled_output)
        sev_logits = self.sev_classifier(pooled_output)
        return MultiTaskOutput(
            syc_logits=syc_logits,
            bias_logits=bias_logits,
            sev_logits=sev_logits,
        )

# ==========================================
# 2. 파일 로드 및 모델 준비
# ==========================================
MODEL_DIR = "."  # /content 루트
TEST_FILE = "sycophancy_test.csv"

if not os.path.exists(TEST_FILE):
    raise FileNotFoundError(f"❌ '{TEST_FILE}' 파일이 존재하지 않습니다. Colab 왼쪽 📁 파일 목록에 업로드했는지 확인해주세요.")

print(f"📂 Test 데이터 로드: {TEST_FILE}")
test_df = pd.read_csv(TEST_FILE)

print("🤖 모델 및 토크나이저 불러오는 중...")
config = AutoConfig.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
eval_model = KcELECTRAMultiTaskModel.from_pretrained(MODEL_DIR, config=config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_model.to(device)
eval_model.eval()

# ==========================================
# 3. Test 데이터 추론 (Inference)
# ==========================================
print("🚀 Test 데이터셋 추론(Inference) 진행 중...")
pred_syc, pred_bias, pred_sev = [], [], []

for text in test_df['text']:
    enc = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = eval_model(**enc)

    pred_syc.append(out.syc_logits.argmax(-1).item())
    pred_bias.append(out.bias_logits.argmax(-1).item())
    pred_sev.append(out.sev_logits.argmax(-1).item())

pred_df = pd.DataFrame({
    'pred_sycophancy': pred_syc,
    'pred_bias_level': pred_bias,
    'pred_severity': pred_sev
})
pred_df.to_csv('model_predictions.csv', index=False, encoding='utf-8-sig')
print("✅ 예측 결과 저장 완료: 'model_predictions.csv'\n")

# ==========================================
# 4. 멀티태스크 종합 평가 및 오답 노트 다운로드
# ==========================================
print("=" * 60)
print("🚀 Sycophancy & Bias Multitask Evaluation Report")
print("=" * 60)

y_true_syc, y_pred_syc = test_df['sycophancy'], pred_df['pred_sycophancy']
y_true_bias, y_pred_bias = test_df['bias_level'], pred_df['pred_bias_level']
y_true_sev, y_pred_sev = test_df['severity'], pred_df['pred_severity']

# 지표 1: Overall Macro F1
print("\n[1. Overall Metrics]")
print(f" - Sycophancy Macro F1 : {f1_score(y_true_syc, y_pred_syc, average='macro'):.4f}")
print(f" - Bias Level Macro F1 : {f1_score(y_true_bias, y_pred_bias, average='macro'):.4f}")
print(f" - Severity Macro F1   : {f1_score(y_true_sev, y_pred_sev, average='macro'):.4f}")
print("-" * 60)

# 지표 2: 클래스 분포 대조
print("\n[2. Class Distribution (Ground Truth vs Prediction)]")
print(f"▶ Sycophancy  GT: {y_true_syc.value_counts().to_dict()} | Pred: {y_pred_syc.value_counts().to_dict()}")
print(f"▶ Bias Level  GT: {y_true_bias.value_counts().to_dict()} | Pred: {y_pred_bias.value_counts().to_dict()}")
print(f"▶ Severity    GT: {y_true_sev.value_counts().to_dict()} | Pred: {y_pred_sev.value_counts().to_dict()}")
print("-" * 60)

# 지표 3: Task별 상세 평가 (CR & CM)
tasks = [
    ("Task 1: Sycophancy (0/1)", y_true_syc, y_pred_syc, ['비동조(0)', '동조(1)']),
    ("Task 2: Bias Level (0~3)", y_true_bias, y_pred_bias, None),
    ("Task 3: Severity (0~2)", y_true_sev, y_pred_sev, None)
]
for title, y_true, y_pred, target_names in tasks:
    print(f"\n[{title}]")
    print("▶ Classification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0))
    print("▶ Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("-" * 60)

# 지표 4: Trap Analysis (역편향 함정 방어력)
print("\n[3. Trap Analysis (지름길 학습 방어력 검증)]")
trap1_mask = (y_true_syc == 0) & (y_true_bias.isin([2, 3]))
trap1_total = trap1_mask.sum()
if trap1_total > 0:
    print(f"▶ Trap 1 (역편향 함정 방어력): 총 {trap1_total}개 데이터")
    print(f"   - Trap Accuracy : {accuracy_score(y_true_syc[trap1_mask], y_pred_syc[trap1_mask]):.4f}")
    print("   - Classification Report (Trap Subset):")
    print(classification_report(y_true_syc[trap1_mask], y_pred_syc[trap1_mask], digits=4, zero_division=0))
else:
    print("▶ Trap 1 대상 데이터가 존재하지 않습니다.")
print("-" * 60)

# 지표 5: Hard Examples 오답 노트 추출 및 자동 다운로드
print("\n[4. Hard Examples (오답 추출)]")
error_mask = (y_true_syc != y_pred_syc) | (y_true_bias != y_pred_bias) | (y_true_sev != y_pred_sev)
errors_df = test_df[error_mask].copy()

if not errors_df.empty:
    errors_df['pred_sycophancy'] = y_pred_syc[error_mask]
    errors_df['pred_bias_level'] = y_pred_bias[error_mask]
    errors_df['pred_severity'] = y_pred_sev[error_mask]

    def get_error_type(row):
        errs = []
        if row['sycophancy'] != row['pred_sycophancy']: errs.append('SYC')
        if row['bias_level'] != row['pred_bias_level']: errs.append('BIAS')
        if row['severity'] != row['pred_severity']: errs.append('SEV')
        return "+".join(errs) + "_ERROR"

    errors_df['error_type'] = errors_df.apply(get_error_type, axis=1)
    cols = ['text', 'error_type', 'sycophancy', 'pred_sycophancy', 'bias_level', 'pred_bias_level', 'severity', 'pred_severity']
    errors_df = errors_df[[c for c in cols if c in errors_df.columns]]

    errors_df.to_csv('hard_examples.csv', index=False, encoding='utf-8-sig')
    print(f"🚨 총 {len(errors_df)}개의 오답 샘플이 발견되었습니다.")
    print("✅ 'hard_examples.csv' 저장 완료! 파일 자동 다운로드를 시작합니다...")
    files.download('hard_examples.csv')
else:
    print("🎉 모든 Test 샘플을 완벽하게 맞췄습니다! (No errors)")

In [ ]:
print("""
### [모델 평가 리포트: sycophancy_multitask_best / 2026-07-23]

1. Overall Performance (전체 성능 요약)
- Sycophancy Macro F1 : 1.0000
- Bias Level Macro F1 : 0.9303
- Severity Macro F1   : 0.9179
- (참고) Sycophancy Accuracy : 1.0000

2. Prediction Distribution (예측 쏠림 여부)
- GT vs Pred 대조 결과: 모드 콜랩스(Mode Collapse) 현상 없이 실제 클래스 비율과 매우 우수하게 일치하여 정상적으로 분포됨 (Bias Level 2와 Severity 1에 미세한 오차 발생)

3. Trap Analysis (함정 방어력)
- Trap 1 (역편향 함정) Accuracy: 1.0000
- (분석: 정답이 비동조(0)이면서 높은 편향을 가진 24개 문장 모두 100% 방어에 성공함. 모델이 혐오/편향 단어에 의존하여 무지성 동조로 오분류하는 Shortcut에 빠지지 않고 문맥 간 관계를 정확히 학습함)

4. Error Analysis (Hard Examples 정성 분석)
- 주요 오답 패턴 1: BIAS+SEV_ERROR가 다수 발생 (Sycophancy 오답은 0건이며, Bias Level 2와 3 간의 경계 구분 오차 위주)
- 주요 오답 패턴 2: Severity는 Bias Level 공식 규칙(severity_mapping)에 따라 종속되므로, Bias Level 오분류 시 Severity 예측도 연쇄적으로 틀어지는 현상 발견
- 조치 계획: 라벨 경계(Bias Level 2 vs 3)에 대한 가이드라인 명확화 및 모호한 샘플 데이터 추가 수집 권장
""")

In [ ]:
from google.colab import files

# 작성한 리포트 내용
report_text = """### [모델 평가 리포트: sycophancy_multitask_best / 2026-07-23]

1. Overall Performance (전체 성능 요약)
- Sycophancy Macro F1 : 1.0000
- Bias Level Macro F1 : 0.9303
- Severity Macro F1   : 0.9179
- (참고) Sycophancy Accuracy : 1.0000

2. Prediction Distribution (예측 쏠림 여부)
- GT vs Pred 대조 결과: 모드 콜랩스(Mode Collapse) 현상 없이 실제 클래스 비율과 매우 우수하게 일치하여 정상적으로 분포됨 (Bias Level 2와 Severity 1에 미세한 오차 발생)

3. Trap Analysis (함정 방어력)
- Trap 1 (역편향 함정) Accuracy: 1.0000
- (분석: 정답이 비동조(0)이면서 높은 편향을 가진 24개 문장 모두 100% 방어에 성공함. 모델이 혐오/편향 단어에 의존하여 무지성 동조로 오분류하는 Shortcut에 빠지지 않고 문맥 간 관계를 정확히 학습함)

4. Error Analysis (Hard Examples 정성 분석)
- 주요 오답 패턴 1: BIAS+SEV_ERROR가 다수 발생 (Sycophancy 오답은 0건이며, Bias Level 2와 3 간의 경계 구분 오차 위주)
- 주요 오답 패턴 2: Severity는 Bias Level 공식 규칙(severity_mapping)에 따라 종속되므로, Bias Level 오분류 시 Severity 예측도 연쇄적으로 틀어지는 현상 발견
- 조치 계획: 라벨 경계(Bias Level 2 vs 3)에 대한 가이드라인 명확화 및 모호한 샘플 데이터 추가 수집 권장
"""

# 1. txt 파일로 저장
with open('evaluation_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)

print("✅ 'evaluation_report.txt' 파일 생성 완료!")

# 2. 내 컴퓨터로 바로 다운로드
files.download('evaluation_report.txt')